# pdb2mcsm — PDB 정리 도구 (Colab)

MD/날것 PDB 복합체를 **mCSM-PPI2가 받아들이는 형태**로 정리합니다.

**자동으로 하는 일**: 사슬 분리(chain ID→TER→번호점프) · 물/이온 제거 · 잔기명 정규화(HIE→HIS 등) · 수소 제거 · 깨끗한 chain ID 부여.
**선택 기능**: 잔기 번호 이동(renumber) · alanine-scanning 목록 생성.

# pdb2mcsm — PDB Processing Tool (Colab)

Processes MD/raw PDB complexes into a **format accepted by mCSM-PPI2**.

**Automated Tasks**: Chain separation (chain ID → TER → number jump) · Water/ion removal · Residue name normalization (HIE → HIS, etc.) · Hydrogen removal · Assignment of clean chain IDs.
**Optional Features**: Residue renumbering · Generation of an alanine-scanning list.


In [ ]:
#@title 1. core.py 가져오기 Importing core.py (GitHub) { display-mode: "form" }

USER = "kjamm"
REPO = "pdb2mcsm"
BRANCH = "main"
PATH_IN_REPO = "core.py"

import urllib.request, os
url = f"https://raw.githubusercontent.com/{USER}/{REPO}/{BRANCH}/{PATH_IN_REPO}"
try:
    urllib.request.urlretrieve(url, "core.py")
    size = os.path.getsize("core.py")
    print(f"✅ core.py 다운로드 완료 ({size} bytes)\n   from {url}")
except Exception as e:
    print(f"❌ 다운로드 실패: {e}\n")
    print("확인: 리포가 public인지, USER/REPO/BRANCH/경로가 맞는지.")
    print(f"URL 직접 확인: {url}")

✅ core.py 다운로드 완료 (8920 bytes)
   from https://raw.githubusercontent.com/kjamm/pdb2mcsm/main/core.py


In [ ]:
#@title 2. 엔진 로드 Load engine
import importlib, core
importlib.reload(core)
print("불러온 함수:", [n for n in dir(core) if not n.startswith('_') and callable(getattr(core,n))])

불러온 함수: ['Atom', 'Chain', 'assign_chain_ids', 'build_ala_scan', 'dataclass', 'field', 'is_hydrogen', 'norm_resname', 'parse_pdb', 'read_atoms', 'renumber_chain', 'summarize', 'write_clean']


In [ ]:
#@title 3. PDB 업로드 upload pdb
from google.colab import files
up = files.upload()
PDB_IN = list(up.keys())[0]
print("업로드됨:", PDB_IN)

Saving 1JDH.pdb to 1JDH.pdb
업로드됨: 1JDH.pdb


In [ ]:
#@title 4. Cleanup + Chain Detection
header, chains, warnings, method = core.parse_pdb(PDB_IN)
core.assign_chain_ids(chains)

print(f"사슬 분리 방법: {method}\n")
print(f"{'chain':<7}{'#res':<7}{'first':<10}{'last':<10}{'split_reason'}")
for cid, n, first, last, reason in core.summarize(chains):
    print(f"{cid:<7}{n:<7}{first:<10}{last:<10}{reason}")

if warnings:
    print("\n⚠️ 경고:")
    for w in warnings:
        print("  -", w)
else:
    print("\n경고 없음")

사슬 분리 방법: existing chain IDs

chain  #res   first     last      split_reason
A      508    ALA135    SER663    existing chain ID
B      38     LEU12     VAL49     existing chain ID

경고 없음


In [ ]:
#@title 5. 한 사슬의 서열·번호 확인 (번호 정할 때 참고) Checking the Sequence and Number of a Chain (for reference when assigning numbers) { display-mode: "form" }
peek_chain = "B"  #@param {type:"string"}
shown = None
for c in chains:
    if c.assigned_chain == peek_chain:
        shown = c; break
if shown is None:
    print("그런 chain 없음. 사용 가능:", [c.assigned_chain for c in chains])
else:
    print(f"chain {peek_chain} ({len(shown.residues)} residues):\n")
    line = ""
    for num, rn, ic in shown.residues:
        line += f"{core.THREE_TO_ONE.get(rn,'X')}{num}{ic.strip()} "
    print(line)

chain B (38 residues):

L12 G13 A14 N15 D16 E17 L18 I19 S20 F21 K22 D23 E24 G25 E26 Q27 E28 E29 K30 S31 S32 E33 N34 S35 S36 A37 E38 R39 D40 L41 A42 D43 V44 K45 S46 S47 L48 V49 


In [ ]:
#@title 6. (선택) 잔기 번호 이동 (Optional) Transfer Remaining Balance { display-mode: "form" }
#@markdown 필요 없으면 이 셀은 건너뛰세요. offset = 논문번호 − 파일번호 (예: 파일 36을 98로 → 62)
#@markdown Skip this cell if it is not needed. offset = paper number − file number (e.g., change file 36 to 98 → 62)
do_renumber = False   #@param {type:"boolean"}
renum_chain = "B"     #@param {type:"string"}
offset = 62           #@param {type:"integer"}

if do_renumber:
    core.renumber_chain(chains, renum_chain, offset)
    print(f"✅ chain {renum_chain} 번호를 {offset:+d} 이동함. 새 범위:")
    for cid, n, first, last, reason in core.summarize(chains):
        if cid == renum_chain:
            print(f"   chain {cid}: {first} .. {last}")
else:
    print("번호 이동 건너뜀 (do_renumber=False)")

번호 이동 건너뜀 (do_renumber=False)


In [ ]:
#@title 7. 깨끗한 PDB 저장 (mCSM에 넣을 파일) Clean PDB File (File to be added to mCSM)
OUT = PDB_IN.rsplit('.',1)[0] + "_ready.pdb"
core.write_clean(OUT, header, chains)
print("✅ 저장:", OUT)
print("   이 파일을 mCSM-PPI2에 업로드하세요.")

✅ 저장: 1JDH_ready.pdb
   이 파일을 mCSM-PPI2에 업로드하세요.


In [ ]:
#@title 8. (선택) alanine-scanning 목록 생성 Creating an alanine-scanning list  { display-mode: "form" }
scan_chain = "A"    #@param {type:"string"}
range_start = 1     #@param {type:"integer"}
range_end = 20      #@param {type:"integer"}

muts = core.build_ala_scan(chains, scan_chain, range_start, range_end)
print("mCSM-PPI2 배치 입력에 붙여넣기:\n")
print("\n".join(muts))
with open("mutations_list.txt", "w") as fh:
    fh.write("\n".join(muts) + "\n")
print("\n저장: mutations_list.txt")

In [ ]:
#@title 9. 다운로드 Download
from google.colab import files
import os
files.download(OUT)
if os.path.exists("mutations_list.txt"):
    files.download("mutations_list.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>